In [7]:
"""
Information Cascade Feature Extraction (For Raw Reddit CSVs)
=============================================================
专门处理zst处理器生成的原始CSV文件

输入: 
- reddit_wsb_posts_raw.csv
- reddit_wsb_comments_raw.csv

输出: 
- cascade_features_5min.parquet
"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from collections import defaultdict, deque

# ============================================================
# 参数设置
# ============================================================

TIME_WINDOW = 5  # 分钟
POSTS_FILE = 'reddit_wsb_posts_raw.csv'
COMMENTS_FILE = 'reddit_wsb_comments_raw.csv'

# ============================================================
# Cascade计算函数（同之前）
# ============================================================

def calculate_structural_virality(edges):
    """计算Structural Virality (Wiener Index)"""
    if len(edges) == 0:
        return 0
    
    graph = defaultdict(list)
    nodes = set()
    
    for parent, child in edges:
        graph[parent].append(child)
        nodes.add(parent)
        nodes.add(child)
    
    nodes = list(nodes)
    n = len(nodes)
    
    if n <= 1:
        return 0
    
    total_distance = 0
    
    for start in nodes:
        distances = {start: 0}
        queue = deque([start])
        
        while queue:
            current = queue.popleft()
            current_dist = distances[current]
            
            for neighbor in graph.get(current, []):
                if neighbor not in distances:
                    distances[neighbor] = current_dist + 1
                    queue.append(neighbor)
        
        total_distance += sum(distances.values())
    
    if n > 1:
        return total_distance / (n * (n - 1))
    else:
        return 0


def calculate_cascade_depth(edges, root_id):
    """计算cascade深度"""
    if len(edges) == 0:
        return 0
    
    graph = defaultdict(list)
    for parent, child in edges:
        graph[parent].append(child)
    
    max_depth = 0
    queue = deque([(root_id, 0)])
    visited = {root_id}
    
    while queue:
        node, depth = queue.popleft()
        max_depth = max(max_depth, depth)
        
        for child in graph.get(node, []):
            if child not in visited:
                visited.add(child)
                queue.append((child, depth + 1))
    
    return max_depth


def calculate_cascade_breadth(edges, root_id):
    """计算cascade最大宽度"""
    if len(edges) == 0:
        return 1
    
    graph = defaultdict(list)
    for parent, child in edges:
        graph[parent].append(child)
    
    level_counts = defaultdict(int)
    queue = deque([(root_id, 0)])
    visited = {root_id}
    
    while queue:
        node, level = queue.popleft()
        level_counts[level] += 1
        
        for child in graph.get(node, []):
            if child not in visited:
                visited.add(child)
                queue.append((child, level + 1))
    
    return max(level_counts.values()) if level_counts else 1


def calculate_adoption_speed(timestamps, root_time):
    """计算传播速度"""
    if len(timestamps) <= 1:
        return 0
    
    times = sorted(timestamps.values())
    duration_hours = (times[-1] - times[0]).total_seconds() / 3600
    
    if duration_hours < 0.01:
        duration_hours = 0.01
    
    return (len(times) - 1) / duration_hours


# ============================================================
# 主函数
# ============================================================

def main():
    start_time = datetime.now()
    
    print("\n" + "="*70)
    print("CASCADE FEATURE EXTRACTION (RAW CSV VERSION)")
    print("="*70)
    
    # ========== Step 1: 加载原始数据 ==========
    print("\nStep 1: Loading raw Reddit data...")
    
    try:
        posts = pd.read_csv(POSTS_FILE)
        posts['timestamp'] = pd.to_datetime(posts['timestamp'])
        print(f"  ✓ Posts: {len(posts):,}")
    except FileNotFoundError:
        print(f"  ✗ {POSTS_FILE} not found!")
        return
    
    try:
        comments = pd.read_csv(COMMENTS_FILE)
        comments['timestamp'] = pd.to_datetime(comments['timestamp'])
        print(f"  ✓ Comments: {len(comments):,}")
    except FileNotFoundError:
        print(f"  ✗ {COMMENTS_FILE} not found!")
        return
    
    # ========== Step 2: 构建Cascade树 ==========
    print("\nStep 2: Building cascade trees...")
    
    cascades = {}
    
    # 为每个post创建cascade
    for _, post in posts.iterrows():
        post_id = post['post_id']
        cascades[post_id] = {
            'nodes': [post_id],
            'edges': [],
            'timestamps': {post_id: post['timestamp']},
            'root_time': post['timestamp']
        }
    
    # 添加comments
    for _, comment in comments.iterrows():
        link_id = comment['post_id']  # 这是root post的ID
        parent_id = comment['parent_id']
        comment_id = comment['comment_id']
        
        if link_id in cascades:
            cascades[link_id]['nodes'].append(comment_id)
            cascades[link_id]['edges'].append((parent_id, comment_id))
            cascades[link_id]['timestamps'][comment_id] = comment['timestamp']
    
    # 只保留有活动的cascades
    active_cascades = {
        k: v for k, v in cascades.items()
        if len(v['nodes']) > 1
    }
    
    print(f"  Total cascades: {len(active_cascades):,}")
    
    # ========== Step 3: 提取特征 ==========
    print("\nStep 3: Extracting cascade features...")
    
    features = []
    
    for post_id, cascade in active_cascades.items():
        nodes = cascade['nodes']
        edges = cascade['edges']
        timestamps = cascade['timestamps']
        root_time = cascade['root_time']
        
        features.append({
            'post_id': post_id,
            'timestamp': root_time,
            'cascade_size': len(nodes),
            'cascade_depth': calculate_cascade_depth(edges, post_id),
            'cascade_breadth': calculate_cascade_breadth(edges, post_id),
            'structural_virality': calculate_structural_virality(edges),
            'adoption_speed': calculate_adoption_speed(timestamps, root_time)
        })
    
    cascade_df = pd.DataFrame(features)
    
    print(f"  ✓ Extracted features for {len(cascade_df):,} cascades")
    print(f"\n  Statistics:")
    print(f"    Mean size: {cascade_df['cascade_size'].mean():.1f}")
    print(f"    Mean depth: {cascade_df['cascade_depth'].mean():.1f}")
    print(f"    Mean virality: {cascade_df['structural_virality'].mean():.2f}")
    
    # ========== Step 4: 聚合到5分钟窗口 ==========
    print("\nStep 4: Aggregating to 5-min windows...")
    
    cascade_df['time_window'] = cascade_df['timestamp'].dt.floor('5min')
    
    agg_features = cascade_df.groupby('time_window').agg({
        'cascade_size': ['mean', 'max', 'sum'],
        'cascade_depth': ['mean', 'max'],
        'cascade_breadth': ['mean', 'max'],
        'structural_virality': ['mean', 'max'],
        'adoption_speed': ['mean', 'max'],
        'post_id': 'count'
    }).reset_index()
    
    agg_features.columns = [
        'timestamp',
        'cascade_size_mean', 'cascade_size_max', 'cascade_size_sum',
        'cascade_depth_mean', 'cascade_depth_max',
        'cascade_breadth_mean', 'cascade_breadth_max',
        'structural_virality_mean', 'structural_virality_max',
        'adoption_speed_mean', 'adoption_speed_max',
        'cascade_count'
    ]
    
    print(f"  ✓ Created {len(agg_features):,} time windows")
    
    # ========== Step 5: 与市场数据对齐 ==========
    print("\nStep 5: Aligning with market data...")
    
    try:
        market = pd.read_csv('gme_5min.csv')
        market['timestamp'] = pd.to_datetime(market['timestamp'], utc=True).dt.tz_localize(None)
        
        # 确保cascade也是tz-naive
        agg_features['timestamp'] = pd.to_datetime(agg_features['timestamp'], utc=True).dt.tz_localize(None)
        
        aligned = market[['timestamp']].merge(
            agg_features,
            on='timestamp',
            how='left'
        )
        
        aligned = aligned.fillna(0)
        
        print(f"  ✓ Aligned to {len(aligned):,} market bars")
        print(f"  Non-zero windows: {(aligned['cascade_count'] > 0).sum():,}")
        
        final_features = aligned
        
    except FileNotFoundError:
        print(f"  ⚠️  gme_5min.csv not found, using unaligned features")
        final_features = agg_features
    
    # ========== Step 6: 保存 ==========
    print("\nStep 6: Saving...")
    
    final_features.to_parquet('cascade_features_5min.parquet', index=False)
    
    print(f"  ✓ Saved: cascade_features_5min.parquet")
    
    # ========== 总结 ==========
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds()
    
    print("\n" + "="*70)
    print("COMPLETE")
    print("="*70)
    print(f"\nCascades: {len(active_cascades):,}")
    print(f"Time windows: {len(final_features):,}")
    print(f"Runtime: {duration:.1f} seconds")
    print("\n✓ Next: python merge_all_features.py")


if __name__ == "__main__":
    main()


CASCADE FEATURE EXTRACTION (RAW CSV VERSION)

Step 1: Loading raw Reddit data...
  ✓ Posts: 211,970
  ✓ Comments: 1,394,123

Step 2: Building cascade trees...
  Total cascades: 27,927

Step 3: Extracting cascade features...
  ✓ Extracted features for 27,927 cascades

  Statistics:
    Mean size: 23.1
    Mean depth: 1.2
    Mean virality: 0.31

Step 4: Aggregating to 5-min windows...
  ✓ Created 12,265 time windows

Step 5: Aligning with market data...
  ✓ Aligned to 59,147 market bars
  Non-zero windows: 6,483

Step 6: Saving...
  ✓ Saved: cascade_features_5min.parquet

COMPLETE

Cascades: 27,927
Time windows: 59,147
Runtime: 43.1 seconds

✓ Next: python merge_all_features.py
